# Alia profile check on a Colab T4 (16 GB)

Measures the `rtx12-16gb-qwen36-35b-q4` profile with its exact `server_args` and context, using the llama.cpp build Alia pins (b11639 = a0e6dc7). Free tier is enough. Run all; the last cell prints the JSONL rows to copy into `bench/results/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!free -g
!df -h /content | tail -1
!nproc

In [ ]:
%%bash
set -e
cd /content
if [ ! -d llama ]; then
  wget -q https://github.com/ggml-org/llama.cpp/releases/download/b11639/llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz
  wget -q https://github.com/ggml-org/llama.cpp/releases/download/b11639/cudart-llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz
  mkdir llama && tar xzf llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz -C llama && tar xzf cudart-llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz -C llama
fi
BIN=$(find /content/llama -name llama-server -type f | head -1)
echo "$BIN"
LD_LIBRARY_PATH=$(dirname $BIN) $BIN --version 2>&1 | tail -2

In [ ]:
%%bash
set -e
pip -q install -U "huggingface_hub[cli]" hf_transfer
cd /content
HF_HUB_ENABLE_HF_TRANSFER=1 hf download unsloth/Qwen3.6-35B-A3B-GGUF Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf --local-dir /content/models
ls -la /content/models
[ -d alia ] || git clone -q --branch micro/r10 https://github.com/BEKO2210/alia
cd alia && git log --oneline -1

In [ ]:
%%bash
cd /content/alia
export LLAMA_SERVER=$(find /content/llama -name llama-server -type f | head -1)
# exact profile: server_args of rtx12-16gb-qwen36-35b-q4 (no -ngl: llama.cpp --fit decides), ctx 32768
python3 bench/bench_llamacpp.py --gguf /content/models/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf --name colab-t4-12-16gb --configs fit --depths 4096,16384,28000 --reps 3 --ctx 32768 --extra "-b 2048 -ub 2048 --jinja --cache-ram 2048" 2>&1 | tail -15

In [ ]:
!for f in /content/alia/bench/results/llamacpp-*.jsonl; do echo "== $f"; cat "$f"; done